# 3. Fair rates, investing, and the people who were turned away (weeks 8-9)

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import plotly.express as px
from lc_rates import config, data, features, finance, model
pd.options.display.float_format = "{:,.4f}".format

In [2]:
df = data.load_clean()
train, test = data.split_train_test(df)
logit, lgbm, meta = model.load_models()
p = model.predict_pd(lgbm, features.build_features(test))
lgd = meta['lgd']
print('Loss given default:', round(lgd, 3))

Loss given default: 0.555


## Expected loss
`expected loss = chance of default × share lost when it happens × amount lent`

Example from the brief: lend $10,000, 10% chance of default, 60% lost → expected loss **$600**.

In [3]:
finance.expected_loss(0.10, 0.60, 10_000)

np.float64(600.0)

## Does each grade's interest pay for its losses?
The last column is what lenders **actually earned per year** (IRR of the real cash flows) after defaults.

In [4]:
finance.grade_economics(test, lgd)

,grade,loans,default_rate,mean_interest_rate,realised_annual_return,expected_loss_share,return_vs_risk_free
0,A,105433,0.0541,0.0709,0.0628,0.0300,0.0428
1,B,145105,0.1146,0.1045,0.0753,0.0637,0.0553
2,C,121390,0.1863,0.1354,0.0720,0.1034,0.0520
3,D,53178,0.2465,0.1679,0.0628,0.1369,0.0428
4,E,16490,0.3035,0.1963,0.0445,0.1685,0.0245
5,F,3338,0.3475,0.2392,0.0435,0.1930,0.0235
6,G,423,0.4208,0.2638,0.0179,0.2337,-0.0021


## The fair rate
`fair rate = risk-free rate + required margin + yearly expected loss`

This is a simplified teaching formula. Change the assumptions and watch the answer move: that sensitivity is part of the finding.

In [5]:
t = test.assign(model_pd=p)
t['fair_rate'] = finance.fair_rate(t['model_pd'], t['term_months'], lgd)
t['gap'] = t['int_rate'] / 100 - t['fair_rate']
by_grade = t.groupby('grade').agg(charged=('int_rate', lambda s: s.mean() / 100), fair=('fair_rate', 'mean'), gap=('gap', 'mean')).reset_index()
px.bar(by_grade.melt('grade', ['charged', 'fair']), x='grade', y='value', color='variable', barmode='group').update_layout(yaxis_tickformat='.0%').show()
by_grade

,grade,charged,fair,gap
0,A,0.0709,0.0657,0.0052
1,B,0.1045,0.0804,0.0241
2,C,0.1354,0.0919,0.0436
3,D,0.1679,0.1004,0.0675
4,E,0.1963,0.1074,0.0888
5,F,0.2392,0.1131,0.1262
6,G,0.2638,0.1211,0.1427


The model-based fair rate **understates risk for the riskiest grades**, because the model's probabilities are squeezed toward the middle. The report also checks it against what lenders really earned. Compare:

In [6]:
pd.read_csv(config.RESULTS_DIR / 'fair_rate_gaps.csv').query("segment == 'grade'")[['value', 'mean_interest_rate', 'mean_fair_rate', 'fair_rate_from_realised_return', 'realised_annual_return']]

,value,mean_interest_rate,mean_fair_rate,fair_rate_from_realised_return,realised_annual_return
0,A,0.0709,0.0657,0.0481,0.0628
1,B,0.1045,0.0804,0.0691,0.0753
2,C,0.1354,0.0919,0.1035,0.0720
3,D,0.1679,0.1004,0.1451,0.0628
4,E,0.1963,0.1074,0.1918,0.0445
5,F,0.2392,0.1131,0.2357,0.0435


## Investing: five strategies on loans nobody had seen

In [7]:
ret_model = model.load_return_model()
pred = ret_model.predict(model.return_features(test))
s = finance.compare_strategies(test, p, share=0.2, n_boot=20, pred_return=pred)
s[['strategy', 'loans', 'annual_return', 'ci95_low', 'ci95_high', 'default_rate', 'mean_interest_rate']]

,strategy,loans,annual_return,ci95_low,ci95_high,default_rate,mean_interest_rate
0,1. Fund every loan,445357,0.0685,0.0679,0.0690,0.1446,0.1171
1,2. Only grade A and B,250538,0.0695,0.0691,0.0700,0.0892,0.0903
2,3. Highest-rate 20% of loans,89072,0.0629,0.0612,0.0645,0.2534,0.1733
3,4. Safest 20% per grade (my model),89069,0.0841,0.0835,0.0851,0.0991,0.1138
4,5. Best 10% by predicted return,44536,0.0911,0.0888,0.0920,0.1729,0.1498


### Design your own rule
Change the mask below and see if you can beat the table. Be honest: tuning a rule on the same loans you judge it on is how people fool themselves. Decide your rule first, test it once.

In [8]:
mask = (test['grade'].isin(['B', 'C'])) & (test['dti'] < 15)   # <- edit me
sel = test[mask]
print(len(sel), 'loans | return', round(finance.pool_irr(sel), 4), '| default rate', round(sel['defaulted'].mean(), 3))

100093 loans | return 0.0822 | default rate 0.125


## The people who were turned away
Only 7.6% of applications were approved. Counts are pre-computed in `app/data/approval_cube.parquet`.

In [9]:
from lc_rates import rejected
cube = pd.read_parquet(config.APP_DATA_DIR / 'approval_cube.parquet')
early = cube[cube['year'].between(2008, 2013)]
rejected.approval_rate(early, 'risk_bin')

,risk_bin,n_accepted,n_rejected,applications,approval_rate
0,600-639,1,258678,258679,0.0000
1,640-659,326,202563,202889,0.0016
2,660-679,62977,182246,245223,0.2568
3,680-699,65316,165173,230489,0.2834
4,700-719,46571,113241,159812,0.2914
5,720-749,34013,75443,109456,0.3107
6,750+,20909,42086,62995,0.3319
7,<600,0,424082,424082,0.0000
8,unknown,0,47647,47647,0.0000


In [10]:
rejected.approval_rate(early, 'dti_bin')

,dti_bin,n_accepted,n_rejected,applications,approval_rate
0,0-10,50723,472954,523677,0.0969
1,10-20,104658,339083,443741,0.2359
2,20-30,65205,258418,323623,0.2015
3,30-40,9527,184877,194404,0.0490
4,40+,0,185414,185414,0.0000
5,invalid (<0),0,70413,70413,0.0000


There is a wall at a risk score of 660 and another at a debt-to-income of 40%. **Careful:** this shows a difference, not why LendingClub decided. Also check what is missing: the rejected file has no income column, and the score scale changes after 2013.

### Your turn
1. Which states have the lowest approval rates? Is that a fairness problem or a mix of applicants?
2. Why do applications with debt-to-income of 0-10% have a *lower* approval rate than 10-20%? (Hint: look at how many have no risk score.)